In [ ]:
import joblib
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

class ApplyKmeans:
    def __init__(self, device):
        self.km_model = joblib.load("/home/alex/Projekt/cfm-vc/ckpt/xeus/kmeans_xeus_500_multilingual.pkl")
        self.C_np = self.km_model.cluster_centers_.transpose()
        self.Cnorm_np = (self.C_np ** 2).sum(0, keepdims=True)
        self.device = device

        self.C = torch.from_numpy(self.C_np).to(device)
        self.Cnorm = torch.from_numpy(self.Cnorm_np).to(device)

    def __call__(self, x):
        x = x.to(self.device)
        dist = (
            x.pow(2).sum(1, keepdim=True)
            - 2 * torch.matmul(x, self.C)
            + self.Cnorm
        )
        return dist.argmin(dim=1).cpu().numpy()

apply_kmeans = ApplyKmeans(device)

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from espnet2.tasks.ssl import SSLTask
import soundfile as sf


xeus_model, xeus_train_args = SSLTask.build_model_from_file(
    '/home/alex/Projekt/cfm-vc/ckpt/xeus/config.yaml',
    '/home/alex/Projekt/cfm-vc/ckpt/xeus/xeus_checkpoint_new.pth',
    device,
)

# INFO_TPL_1441_FIRSTWARN_CONDITION_LESTER_15_01.mp3
# INFO_CORANGAR_FINDHERB_SUCCESS_08_02.mp3
wavs, sampling_rate = sf.read('/home/alex/Data/Gothic 1/deu/SPEECH/INFO_TPL_1441_FIRSTWARN_CONDITION_LESTER_15_01.mp3') # sampling rate should be 16000
wav_lengths = torch.LongTensor([len(wav) for wav in [wavs]]).to(device)
wavs = pad_sequence(torch.Tensor([wavs]), batch_first=True).to(device) 

# we recommend use_mask=True during fine-tuning
feats = xeus_model.encode(wavs, wav_lengths, use_final_output=False)


In [ ]:
features = feats["encoder_output"][14].squeeze(0)

In [ ]:
units = apply_kmeans(features).tolist()
units